# Data Transformation Techniques in Machine Learning

**Based on the lecture PDF**: Mapping, Quantization, Binarization, Normalization, Standardization (Scaling), Robust Scaling

### What you will learn
- Why and where transformations fit in the pipeline
- How to implement each technique with `pandas` and `scikit-learn`
- How to combine transformations with `ColumnTransformer` and `Pipeline`
- How to verify and invert transforms where applicable


## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    OrdinalEncoder, OneHotEncoder, KBinsDiscretizer,
    Binarizer, MinMaxScaler, StandardScaler, RobustScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn import set_config
set_config(transform_output='pandas')
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.__version__

'2.2.2'

## 2. Create a synthetic dataset
We create a small, realistic dataset that contains numeric features, categorical features, and some outliers. This will let us demonstrate each transformation clearly.

In [ ]:
n = 40
age = np.random.randint(16, 70, size=n)
height_cm = np.random.normal(loc=165, scale=10, size=n).round(1)
income_k = np.random.normal(loc=55, scale=18, size=n).round(1)
education = np.random.choice(['High School', 'Bachelor', 'Master'], size=n, p=[0.35, 0.45, 0.20])
gender = np.random.choice(['Female', 'Male'], size=n, p=[0.45, 0.55])
score = np.random.randint(0, 101, size=n)

# inject a few outliers
income_k[:3] = [5, 200, 1]  # extreme values
height_cm[:2] = [130, 210]  # extreme heights

df = pd.DataFrame({
    'age': age,
    'height_cm': height_cm,
    'income_k': income_k,
    'education': education,
    'gender': gender,
    'score': score
})
df.head()

,age,height_cm,income_k,education,gender,score
0,54,130.0,5.0,Master,Female,53
1,67,210.0,200.0,High School,Male,32
2,44,176.3,1.0,High School,Male,23
3,30,164.0,58.1,Bachelor,Male,74
4,58,159.7,66.9,Master,Female,71


## 3. Where transformations fit in the pipeline
Raw Data → Cleaning → **Transformation** → Modeling

In this notebook we focus on the transformation block.


## 4. Mapping (Label or Ordinal Encoding)
Mapping converts categorical text to numeric. Use it with care. If the categories do not have a true order, prefer One Hot Encoding for models that use distance or linearity.

In [ ]:
# Example A: simple mapping with pandas .map for binary gender
map_gender = {'Female': 0, 'Male': 1}
df['gender_mapped'] = df['gender'].map(map_gender)
df[['gender', 'gender_mapped']].head()


,gender,gender_mapped
0,Female,0
1,Male,1
2,Male,1
3,Male,1
4,Female,0


In [ ]:
# Example B: Ordinal encoding for education (Low < Mid < High)
edu_order = [['High School', 'Bachelor', 'Master']]
ord_enc = OrdinalEncoder(categories=edu_order)
df['education_ord'] = ord_enc.fit_transform(df[['education']])
df[['education', 'education_ord']].head()

,education,education_ord
0,Master,2.0
1,High School,0.0
2,High School,0.0
3,Bachelor,1.0
4,Master,2.0


In [ ]:
# Example C (optional): One Hot Encoding when order is not meaningful
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_df = pd.DataFrame(
    ohe.fit_transform(df[['education']]),
    columns=ohe.get_feature_names_out(['education'])
)
pd.concat([df[['education']], ohe_df], axis=1).head()

,education,education_Bachelor,education_High School,education_Master
0,Master,0.0,0.0,1.0
1,High School,0.0,1.0,0.0
2,High School,0.0,1.0,0.0
3,Bachelor,1.0,0.0,0.0
4,Master,0.0,0.0,1.0


## 5. Quantization (Binning)
Quantization turns continuous values into discrete bins. Useful for rule-based models and to simplify ranges.

In [ ]:
# Example A: custom bins for age
age_bins = [0, 12, 19, 59, np.inf]
age_labels = ['Child', 'Teen', 'Adult', 'Senior']
df['age_bin'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=True)
df[['age', 'age_bin']].head()

,age,age_bin
0,54,Adult
1,67,Senior
2,44,Adult
3,30,Adult
4,58,Adult


In [ ]:
# Example B: data-driven equal-width bins using KBinsDiscretizer
kbin = KBinsDiscretizer(n_bins=6, encode='ordinal', strategy='uniform')
df['income_bin_uniform'] = kbin.fit_transform(df[['income_k']])
df[['income_k', 'income_bin_uniform']].head()

,income_k,income_bin_uniform
0,5.0,0.0
1,200.0,5.0
2,1.0,0.0
3,58.1,1.0
4,66.9,1.0


## 6. Binarization
Binarization converts values to 0 or 1 via a threshold.

In [ ]:
# Example A: simple threshold with pandas
threshold = 50
df['high_income'] = (df['income_k'] > threshold).astype(int)
df[['income_k', 'high_income']].head()

,income_k,high_income
0,5.0,0
1,200.0,1
2,1.0,0
3,58.1,1
4,66.9,1


In [ ]:
# Example B: scikit-learn Binarizer
binzr = Binarizer(threshold=threshold)
df['score_pass'] = binzr.fit_transform(df[['score']])
df[['score', 'score_pass']].head()

,score,score_pass
0,53,1
1,32,0
2,23,0
3,74,1
4,71,1


## 7. Normalization (Min-Max Scaling)
Rescales selected features to [0, 1]. Helps when features must share a bounded range.

In [ ]:
mm_cols = ['age', 'height_cm', 'income_k']
mm = MinMaxScaler()
mm_df = mm.fit_transform(df[mm_cols]).add_suffix('_mm')  # already a DataFrame
pd.concat([df[mm_cols].head(), mm_df.head()], axis=1)

,age,height_cm,income_k,age_mm,height_cm_mm,income_k_mm
0,54,130.0,5.0,0.725490,0.00000,0.020101
1,67,210.0,200.0,0.980392,1.00000,1.000000
2,44,176.3,1.0,0.529412,0.57875,0.000000
3,30,164.0,58.1,0.254902,0.42500,0.286935
4,58,159.7,66.9,0.803922,0.37125,0.331156


## 8. Scaling (Standardization)
Centers features by mean and scales to unit variance. Often preferred by linear models and SVM.

In [ ]:
std_cols = ['age', 'height_cm', 'income_k']
scaler = StandardScaler()
std_df = scaler.fit_transform(df[std_cols]).add_suffix('_z')  # DataFrame with suffix
std_df.describe().loc[['mean','std']]

,age_z,height_cm_z,income_k_z
mean,2.220446e-16,5.995204e-16,8.881784e-17
std,1.012739e+00,1.012739e+00,1.012739e+00


## 9. Robust Scaling
Centers by median and scales by interquartile range (IQR). Good when there are outliers.

In [ ]:
rob_cols = ['age', 'height_cm', 'income_k']
rob = RobustScaler()
rob_df = rob.fit_transform(df[rob_cols]).add_suffix('_rob')  # DataFrame with suffix
rob_df.describe().round(2)

,age_rob,height_cm_rob,income_k_rob
count,40.00,40.00,40.00
mean,0.11,0.09,-0.07
std,0.67,0.90,1.06
min,-1.08,-2.28,-1.96
25%,-0.35,-0.43,-0.68
50%,0.00,-0.00,0.00
75%,0.65,0.57,0.32
max,1.21,3.03,4.91


## 10. Putting it all together with ColumnTransformer and Pipeline
We combine different transforms per column group and return a single transformed DataFrame.

In [ ]:
num_cols = ['age', 'height_cm', 'income_k']
cat_ord_cols = ['education']  # with a known order
cat_nom_cols = ['gender']     # no natural order

pre = ColumnTransformer(
    transformers=[
        ('num_mm', MinMaxScaler(), num_cols),
        ('edu_ord', OrdinalEncoder(categories=[['High School','Bachelor','Master']]), cat_ord_cols),
        ('gender_ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_nom_cols)
    ],
    remainder='drop'
)

X_trans = pre.fit_transform(df)
X_trans.head()

,num_mm__age,num_mm__height_cm,num_mm__income_k,edu_ord__education,gender_ohe__gender_Female,gender_ohe__gender_Male
0,0.725490,0.00000,0.020101,2.0,1.0,0.0
1,0.980392,1.00000,1.000000,0.0,0.0,1.0
2,0.529412,0.57875,0.000000,0.0,0.0,1.0
3,0.254902,0.42500,0.286935,1.0,0.0,1.0
4,0.803922,0.37125,0.331156,2.0,1.0,0.0


## 11. Verifying ranges and inverse transforming
We check ranges for Min-Max scaled columns and show how to invert a scaler. Only invertible parts can be reversed with the attached transformer.

In [ ]:
# Verify ranges of Min-Max features
X_num_mm = X_trans.filter(like='num_mm__')
X_num_mm.agg(['min','max']).round(3)

,num_mm__age,num_mm__height_cm,num_mm__income_k
min,0.0,0.0,0.0
max,1.0,1.0,1.0


In [ ]:
# Example of inverse_transform for MinMaxScaler alone
mm_only = MinMaxScaler().fit(df[num_cols])
mm_values = mm_only.transform(df[num_cols])
restored = mm_only.inverse_transform(mm_values)
pd.DataFrame(restored, columns=num_cols).head()

,age,height_cm,income_k
0,54.0,130.0,5.0
1,67.0,210.0,200.0
2,44.0,176.3,1.0
3,30.0,164.0,58.1
4,58.0,159.7,66.9


## 12. Practice snippets (from the PDF)
These short code blocks align with the coding challenges in the lecture slides.

In [ ]:
# Q13: Map education strings to [1, 2, 3]
edu = pd.Series(['High School', 'Bachelor', 'Master'])
edu_map = {'High School':1, 'Bachelor':2, 'Master':3}
edu_mapped = edu.map(edu_map)
edu.to_frame('education').assign(mapped=edu_mapped)

,education,mapped
0,High School,1
1,Bachelor,2
2,Master,3


In [ ]:
# Q14: Min-Max normalize [10, 20, 30, 40]
arr = np.array([[10],[20],[30],[40]])
mm2 = MinMaxScaler()
mm2.fit_transform(arr).values.ravel()

array([0.        , 0.33333333, 0.66666667, 1.        ])

In [ ]:
# Q15: Standardize [5, 10, 15]
arr = np.array([[5],[10],[15]])
StandardScaler().fit_transform(arr).values.ravel()

array([-1.22474487,  0.        ,  1.22474487])

In [ ]:
# Q16: Robust scaling [1, 100, 3, 2, 5, 1000]
arr = np.array([[1],[100],[3],[2],[5],[1000]])
RobustScaler().fit_transform(arr).values.ravel()

array([-0.04054054,  1.2972973 , -0.01351351, -0.02702703,  0.01351351,
       13.45945946])

In [ ]:
# Q17: Binarize a 'score' column with threshold 50
df_temp = pd.DataFrame({'score':[10, 49, 50, 51, 99]})
df_temp['pass'] = (df_temp['score'] > 50).astype(int)
df_temp

,score,pass
0,10,0
1,49,0
2,50,0
3,51,1
4,99,1


In [ ]:
# Q18: Quantize temperature into Cold, Warm, Hot
def temp_to_label(t):
    if t < 15:
        return 'Cold'
    elif t < 30:
        return 'Warm'
    else:
        return 'Hot'

temps = pd.Series([5, 14.9, 15, 22, 29.9, 30, 41])
temps.to_frame('temp_C').assign(label=temps.apply(temp_to_label))

,temp_C,label
0,5.0,Cold
1,14.9,Cold
2,15.0,Warm
3,22.0,Warm
4,29.9,Warm
5,30.0,Hot
6,41.0,Hot


## 13. Wrap up
- Mapping, quantization, and binarization change the representation or granularity of data.
- Normalization and scaling change the numeric range and distribution shape.
- Robust scaling helps when outliers exist.
- ColumnTransformer lets you apply different transforms per column in a single pipeline.

**##Step-by-step calculation of correlation coefficient**

In [ ]:
import pandas as pd
import numpy as np

# Small dataset
x = np.array([2, 4, 6])
y = np.array([5, 9, 12])

# Step 1: Calculate means
x_bar = np.mean(x)
y_bar = np.mean(y)

# Step 2: Calculate deviations
x_dev = x - x_bar
y_dev = y - y_bar

# Step 3: Multiply deviations
product = x_dev * y_dev

# Step 4: Calculate squares for std
x_sq = x_dev ** 2
y_sq = y_dev ** 2

# Step 5: Compute sums
sum_xy = np.sum(product)
sum_x2 = np.sum(x_sq)
sum_y2 = np.sum(y_sq)

# Step 6: Compute standard deviations
sx = np.sqrt(sum_x2 / (len(x) - 1))
sy = np.sqrt(sum_y2 / (len(y) - 1))

# Step 7: Compute correlation coefficient
r = sum_xy / ((len(x) - 1) * sx * sy)

# Display step-by-step results
df = pd.DataFrame({
    'x': x,
    'y': y,
    '(x - x̄)': x_dev.round(2),
    '(y - ȳ)': y_dev.round(2),
    '(x - x̄)(y - ȳ)': product.round(2),
    '(x - x̄)²': x_sq.round(2),
    '(y - ȳ)²': y_sq.round(2)
})

print("Step-by-step calculation of correlation coefficient:\n")
print(df)
print("\nMeans: x̄ =", round(x_bar,2), ", ȳ =", round(y_bar,2))
print("Standard Deviations: sx =", round(sx,2), ", sy =", round(sy,2))
print("\nSum of (x - x̄)(y - ȳ) =", round(sum_xy,2))
print("\nCorrelation coefficient (r) =", round(r,3))

Step-by-step calculation of correlation coefficient:

   x   y  (x - x̄)  (y - ȳ)  (x - x̄)(y - ȳ)  (x - x̄)²  (y - ȳ)²
0  2   5      -2.0    -3.67             7.33        4.0     13.44
1  4   9       0.0     0.33             0.00        0.0      0.11
2  6  12       2.0     3.33             6.67        4.0     11.11

Means: x̄ = 4.0 , ȳ = 8.67
Standard Deviations: sx = 2.0 , sy = 3.51

Sum of (x - x̄)(y - ȳ) = 14.0

Correlation coefficient (r) = 0.997
